In [0]:
from pyspark.sql.functions import current_timestamp, lit
import uuid


# ============================================
# 1. CREATE INGESTION RUN METADATA
# ============================================

batch_id = str(uuid.uuid4())
ingestion_timestamp = current_timestamp()

print(f"Batch ID: {batch_id}")

In [0]:
# ============================================
# 2. PREPARE CUSTOMERS FOR BRONZE
# ============================================

customer_bronze = (
    customer_df
    .withColumn("_ingestion_timestamp", ingestion_timestamp)
    .withColumn("_source", lit("CRM"))
    .withColumn("_batch_id", lit(batch_id))
)


# ============================================
# 3. PREPARE PRODUCTS FOR BRONZE
# ============================================

product_bronze = (
    product_df
    .withColumn("_ingestion_timestamp", ingestion_timestamp)
    .withColumn("_source", lit("ERP"))
    .withColumn("_batch_id", lit(batch_id))
)


# ============================================
# 4. PREPARE ORDERS FOR BRONZE
# ============================================

orders_bronze = (
    orders_df
    .withColumn("_ingestion_timestamp", ingestion_timestamp)
    .withColumn("_source", lit("ERP"))
    .withColumn("_batch_id", lit(batch_id))
)

In [0]:
customer_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_customers")

product_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_products")

orders_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_sales_orders")

print("Bronze ingestion completed successfully.")
print(f"Batch ID: {batch_id}")

In [0]:
print("Customers:", spark.table("bronze_customers").count())
print("Products:", spark.table("bronze_products").count())
print("Orders:", spark.table("bronze_sales_orders").count())

In [0]:

display(
    spark.table("bronze_customers")
    .select(
        "_batch_id",
        "_source",
        "_ingestion_timestamp"
    )
    .distinct()
)

display(
    spark.table("bronze_products")
    .select(
        "_batch_id",
        "_source",
        "_ingestion_timestamp"
    )
    .distinct()
)

display(
    spark.table("bronze_sales_orders")
    .select(
        "_batch_id",
        "_source",
        "_ingestion_timestamp"
    )
    .distinct()
)